# Local ranked recovery experiment — no Z3

100 synthetic cases comparing random placement, a weighted score, and ordinary load balancing; each with uniform or selective output checkpointing. No cloud calls, downloads, or real ML training. Only planning time is measured; application times and costs are modeled.

Read `selective_replication/data_processing/RANKED_SIMULATION.md` for the score, controls and failure model. All code/settings below are editable. Rerun the run cell to resume the same suite. Change the suite name after changing settings or source.


## 1. Paths and settings


In [ ]:
from pathlib import Path
import json
import sys

# Works when Jupyter starts in the repository or a directory beneath it.
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'gossip_benchmarks/selective_replication/data_processing/ranked_simulation.py').exists())
MODULE_DIR = REPO / 'gossip_benchmarks/selective_replication/data_processing'
sys.path.insert(0, str(MODULE_DIR))

SUITE = 'ranked-local-v1'
OUTPUT = REPO / 'ray_gcp_results' / SUITE
MAX_EXPERIMENT_MINUTES = 10  # checked between cases; maximum 28
CONFIG = {
    'cases': 100,                 # multiple of 5: equal numbers per regime
    'seed': 20261027,             # freeze before inspecting outcomes
    'random_repeats': 10,         # averaged within each case
    'weights': {'time': 1.0, 'risk': 1.0, 'cost': 1.0},
    'bootstrap_repeats': 1000,
}
print('Output:', OUTPUT)
print(json.dumps(CONFIG, indent=2))


## 2. Run or resume

No winner is assumed. Completed cases are reused; an interrupted case starts again. Do not start two controllers on the same output directory. The time budget excludes summary/archive writing. A source/settings/Python change requires a new suite name.


In [ ]:
from ranked_simulation import run

summary = run(OUTPUT, config=CONFIG, minutes=MAX_EXPERIMENT_MINUTES)
print('Finished:', summary['finished'])
print('Cases:', summary['completed_cases'], '/', summary['planned_cases'])


## 3. Read results without rerunning the experiment

Positive percentage means the selected method is better. All cases enter the comparisons, including infeasible plans; check their feasibility fractions. Random placement trials are averaged within each case. The 95% bootstrap intervals are descriptive, not proof of a general advantage.


In [ ]:
summary = json.loads((OUTPUT / 'summary.json').read_text())
print('Completed:', summary['completed_cases'], '/', summary['planned_cases'])
for row in summary['comparisons']:
    if row['regime'] == 'all':
        print(f"{row['selected']} vs {row['baseline']} | {row['metric']} | "
              f"mean gain {row['mean_gain_percent']:.2f}% | "
              f"wins/ties/losses {row['wins']}/{row['ties']}/{row['losses']} | "
              f"feasible {row['selected_feasible_fraction']:.0%}/{row['baseline_feasible_fraction']:.0%}")


## 4. Plot modeled time and cost gains

Dots show case-averaged gains; horizontal intervals show a descriptive 95% bootstrap interval. Zero means a tie; left of zero is worse. Change `REGIME` to any regime in the generator to inspect where the rule loses. `planning_inclusive` adds measured local planning time to modeled execution.


In [ ]:
import matplotlib.pyplot as plt

REGIME = 'all'  # homogeneous_low_risk, independent, cheap_risky, fast_risky, checkpoint_expensive
TIME_METRIC = 'planning_inclusive'  # or 'time' for modeled execution alone
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, metric, title in zip(axes, [TIME_METRIC, 'cost'],
                              ['Time reduction (%)', 'Modeled rental-cost reduction (%)']):
    rows = [r for r in summary['comparisons'] if r['regime'] == REGIME and r['metric'] == metric]
    for y, row in enumerate(rows):
        value = row['mean_gain_percent']
        bounds = row['bootstrap_95_percent']
        color = 'tab:blue' if value >= 0 else 'tab:red'
        if bounds is not None:
            ax.hlines(y, bounds[0], bounds[1], color=color, linewidth=2)
        ax.scatter(value, y, color=color, s=40, zorder=3)
    ax.set_yticks(range(len(rows)), [r['selected'] + ' vs ' + r['baseline'] for r in rows])
    ax.set_title(title)
    ax.xaxis.tick_top()
    ax.tick_params(bottom=False, labelbottom=False)
    ax.spines[['bottom', 'right']].set_visible(False)
axes[0].invert_yaxis()
fig.suptitle(f"{REGIME}: {summary['completed_cases']} simulated cases", y=1.02)
fig.tight_layout()
plt.show()


## 5. Files to share

`analysis.zip` includes all cases, assumptions, assignments, timing measurements and source snapshots. Preserve negative outcomes. If tuning the weights, use a separate development run and freeze them before a new evaluation seed.


In [ ]:
print('Upload:', OUTPUT / 'analysis.zip')
print('Summary:', OUTPUT / 'summary.md')
print('All observations:', OUTPUT / 'results.csv')
